In [ ]:
!pip install torch==2.4.1 torchvision==0.19.1 --quiet
!pip install diffusers==0.30.0 transformers==4.44.0 accelerate controlnet-aux xformers opencv-python --quiet

In [ ]:
from google.colab import auth
auth.authenticate_user()

!mkdir -p /content/images
!gsutil -m cp gs://psoriasis_bucket/dataset/images/*.jpeg /content/images/
!ls /content/images

In [ ]:
import torch
import os
from PIL import Image
from controlnet_aux import HEDdetector
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from diffusers.utils import load_image

version = "v2"  # change this each run
os.makedirs(f"/content/outputs/{version}", exist_ok=True)

prompts = {
    "34deac35a907f46e6d43b7bfb39fc00d.jpeg": (
        "medical realism clinical photograph, dark skin tone, multiple discrete psoriatic plaques on torso, thick white silvery scales, well-defined borders, natural studio lighting, dermatology reference photo",
        "blurry, low quality, distorted, watermark, text, logo, illustration, drawing, extra limbs, unnatural colors"
    ),
    "40c298a68522a302b8a22ebf3395194e.jpeg": (
        "medical realism clinical photograph, light skin tone, severe plaque psoriasis covering entire forearm, dense reddish inflamed skin, thick layered scaling, body hair visible, dermatology reference photo",
        "blurry, low quality, distorted, watermark, text, logo, illustration, smooth skin, healthy skin, extra limbs"
    ),
    "56783377d95a87b04c0f10ec52e0ffa7.jpeg": (
        "medical realism clinical photograph, extreme close up of limb, light to medium skin tone, heavy psoriatic scaling with tan and brown crusting, erythematous base visible beneath scales, clinical dermatology reference",
        "blurry, low quality, distorted, watermark, text, logo, illustration, smooth skin, healthy skin"
    ),
    "b4b0eeeb1cfc1b7fa8cae06c3bfd3cb8.jpeg": (
        "medical realism clinical photograph, dark skin tone, three fingers close up, psoriatic nail dystrophy with pitting and discoloration, scaly plaques on finger joints, white fabric background, clinical dermatology reference",
        "blurry, low quality, distorted, watermark, text, logo, rings, jewelry, extra fingers, deformed anatomy, smooth nails"
    ),
    "c7c93112a8dd45edeacfe3496ff8a4c9.jpeg": (
        "medical realism clinical photograph, dark skin tone, dorsal view of both hands side by side, psoriatic plaques across knuckles and fingers, inflamed red and scaly lesions, watch visible on wrist, clinical dermatology reference",
        "blurry, low quality, distorted, watermark, text, logo, single hand, healthy skin, smooth skin, extra fingers"
    ),
}

processor = HEDdetector.from_pretrained('lllyasviel/Annotators')
controlnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_softedge", torch_dtype=torch.float16)
pipe = StableDiffusionControlNetPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", controlnet=controlnet, torch_dtype=torch.float16)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.safety_checker = None  # disable for clinical/medical images
pipe.enable_model_cpu_offload()

for img_file, (prompt, negative_prompt) in prompts.items():
    img_path = f"/content/images/{img_file}"
    if not os.path.exists(img_path):
        print(f"Skipping {img_file} — not found")
        continue
    image = load_image(img_path).resize((512, 512))
    softedge = processor(image)
    output = pipe(
        prompt,
        image=softedge,
        negative_prompt=negative_prompt,
        num_inference_steps=50,
        guidance_scale=9.0,
        controlnet_conditioning_scale=0.6
    ).images[0]
    output.save(f"/content/outputs/{version}/{img_file}")
    print(f"Done: {img_file}")

print("All images complete!")

In [ ]:
#v3
import torch
import os
from PIL import Image
from controlnet_aux import HEDdetector
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from diffusers.utils import load_image

version = "v3"
os.makedirs(f"/content/outputs/{version}/all", exist_ok=True)

prompts = {
    "34deac35a907f46e6d43b7bfb39fc00d.jpeg": (
        "An image of psoriasis on the torso of a dark skinned person",
        "blurry, low quality, distorted, watermark, text, logo, illustration, drawing, unnatural colors"
    ),
    "40c298a68522a302b8a22ebf3395194e.jpeg": (
        "An image of psoriasis on the forearm of a light skinned person",
        "blurry, low quality, distorted, watermark, text, logo, illustration, smooth skin, healthy skin"
    ),
    "56783377d95a87b04c0f10ec52e0ffa7.jpeg": (
        "An image of psoriasis on the leg of a medium skinned person",
        "blurry, low quality, distorted, watermark, text, logo, illustration, smooth skin, healthy skin"
    ),
    "b4b0eeeb1cfc1b7fa8cae06c3bfd3cb8.jpeg": (
        "An image of psoriasis on the fingers of a dark skinned person",
        "blurry, low quality, distorted, watermark, text, logo, rings, jewelry, extra fingers, deformed anatomy"
    ),
    "c7c93112a8dd45edeacfe3496ff8a4c9.jpeg": (
        "An image of psoriasis on the hands of a dark skinned person",
        "blurry, low quality, distorted, watermark, text, logo, single hand, healthy skin, smooth skin, extra fingers"
    ),
}

processor = HEDdetector.from_pretrained('lllyasviel/Annotators')
controlnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_softedge", torch_dtype=torch.float16)
pipe = StableDiffusionControlNetPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", controlnet=controlnet, torch_dtype=torch.float16)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.safety_checker = None
pipe.enable_model_cpu_offload()

for img_file, (prompt, negative_prompt) in prompts.items():
    img_path = f"/content/images/{img_file}"
    if not os.path.exists(img_path):
        print(f"Skipping {img_file} — not found")
        continue

    # tight center crop around pathology
    image = load_image(img_path)
    w, h = image.size
    crop_size = min(w, h)
    left = (w - crop_size) // 2
    top = (h - crop_size) // 2
    image = image.crop((left, top, left + crop_size, top + crop_size)).resize((512, 512))

    softedge = processor(image)

    # generate 4 images per seed
    outputs = pipe(
        prompt,
        image=softedge,
        negative_prompt=negative_prompt,
        num_inference_steps=50,
        guidance_scale=9.0,
        controlnet_conditioning_scale=0.6,
        num_images_per_prompt=4
    ).images

    # save into individual image folder and all folder
    img_name = img_file.replace(".jpeg", "")
    img_folder = f"/content/outputs/{version}/{img_name}"
    os.makedirs(img_folder, exist_ok=True)

    for i, output in enumerate(outputs):
        filename = f"{img_name}_gen{i+1}.jpeg"
        output.save(f"{img_folder}/{filename}")
        output.save(f"/content/outputs/{version}/all/{filename}")
        print(f"Saved: {filename}")

    print(f"Done: {img_file}")

print("All images complete!")

In [ ]:
#v4
import torch
import os
import zipfile
from PIL import Image
from diffusers import StableDiffusionImg2ImgPipeline
from diffusers.utils import load_image

version = "v4"
os.makedirs(f"/content/outputs/{version}/all", exist_ok=True)

prompts = {
    "34deac35a907f46e6d43b7bfb39fc00d.jpeg": (
        "a highly detailed, medical realism close up photo of a torso, dark skin tone, multiple distinct scaly psoriatic plaques, thick silvery white scales, well defined lesion borders, clear clinical lighting, dermatology photography",
        "blurry, low quality, distorted, watermark, text, logo, illustration, drawing, unnatural colors, healthy skin, smooth skin"
    ),
    "40c298a68522a302b8a22ebf3395194e.jpeg": (
        "a highly detailed, medical realism close up photo of a forearm, light skin tone, severe psoriatic plaques covering the skin, dense reddish inflamed lesions, thick layered scaling, body hair visible, clear clinical lighting, dermatology photography",
        "blurry, low quality, distorted, watermark, text, logo, illustration, smooth skin, healthy skin"
    ),
    "56783377d95a87b04c0f10ec52e0ffa7.jpeg": (
        "a highly detailed, medical realism close up photo of a leg, medium skin tone, heavy psoriatic scaling, tan and brown crusting, erythematous base visible beneath scales, clear clinical lighting, dermatology photography",
        "blurry, low quality, distorted, watermark, text, logo, illustration, smooth skin, healthy skin"
    ),
    "b4b0eeeb1cfc1b7fa8cae06c3bfd3cb8.jpeg": (
        "a highly detailed, medical realism close up photo of three fingers, dark skin tone, psoriatic nail dystrophy with pitting and discoloration, distinct scaly plaques on finger joints, natural fingernails, white fabric background, clear clinical lighting, dermatology photography",
        "blurry, low quality, distorted, watermark, text, logo, rings, jewelry, extra fingers, deformed anatomy, smooth nails, low resolution"
    ),
    "c7c93112a8dd45edeacfe3496ff8a4c9.jpeg": (
        "a highly detailed, medical realism close up photo of two hands, dark skin tone, psoriatic plaques across knuckles and fingers, inflamed red and scaly lesions, natural fingernails, clear clinical lighting, dermatology photography",
        "blurry, low quality, distorted, watermark, text, logo, single hand, healthy skin, smooth skin, extra fingers, low resolution"
    ),
}

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)
pipe.safety_checker = None
pipe.enable_model_cpu_offload()

for img_file, (prompt, negative_prompt) in prompts.items():
    img_path = f"/content/images/{img_file}"
    if not os.path.exists(img_path):
        print(f"Skipping {img_file} — not found")
        continue

    # tight center crop
    image = load_image(img_path)
    w, h = image.size
    crop_size = min(w, h)
    left = (w - crop_size) // 2
    top = (h - crop_size) // 2
    image = image.crop((left, top, left + crop_size, top + crop_size)).resize((512, 512))

    # generate 4 images per seed
    outputs = pipe(
        prompt=prompt,
        image=image,
        negative_prompt=negative_prompt,
        strength=0.4,
        guidance_scale=9.0,
        num_inference_steps=50,
        num_images_per_prompt=4
    ).images

    # save into individual folder and all folder
    img_name = img_file.replace(".jpeg", "")
    img_folder = f"/content/outputs/{version}/{img_name}"
    os.makedirs(img_folder, exist_ok=True)

    for i, output in enumerate(outputs):
        filename = f"{img_name}_gen{i+1}.jpeg"
        output.save(f"{img_folder}/{filename}")
        output.save(f"/content/outputs/{version}/all/{filename}")
        print(f"Saved: {filename}")

    print(f"Done: {img_file}")

print("All images complete!")

In [ ]:
import torch
import os
import zipfile
from PIL import Image
from diffusers import StableDiffusionImg2ImgPipeline
from diffusers.utils import load_image

version = "v5"
os.makedirs(f"/content/outputs/{version}/all", exist_ok=True)

prompts = {
    "34deac35a907f46e6d43b7bfb39fc00d.jpeg": (
        "a highly detailed medical realism photograph of a dark skinned torso, multiple raised white silvery psoriatic plaques, scaly patches with defined borders on dark brown skin, natural skin texture visible between lesions, clinical dermatology photography",
        "blurry, low quality, distorted, watermark, text, logo, SFS, letter S, illustration, drawing, unnatural colors, healthy skin, smooth skin"
    ),
    "40c298a68522a302b8a22ebf3395194e.jpeg": (
        "a highly detailed, medical realism close up photo of a forearm, light skin tone, severe psoriatic plaques covering the skin, dense reddish inflamed lesions, thick layered scaling, body hair visible, clear clinical lighting, dermatology photography",
        "blurry, low quality, distorted, watermark, text, logo, SFS, illustration, smooth skin, healthy skin"
    ),
    "56783377d95a87b04c0f10ec52e0ffa7.jpeg": (
        "a highly detailed, medical realism close up photo of a leg, medium skin tone, heavy psoriatic scaling, tan and brown crusting, erythematous base visible beneath scales, clear clinical lighting, dermatology photography",
        "blurry, low quality, distorted, watermark, text, logo, SFS, illustration, smooth skin, healthy skin"
    ),
    "b4b0eeeb1cfc1b7fa8cae06c3bfd3cb8.jpeg": (
        "a highly detailed, medical realism close up photo of three fingers, dark skin tone, psoriatic nail dystrophy with pitting and discoloration, distinct scaly plaques on finger joints, natural fingernails, white fabric background, clear clinical lighting, dermatology photography",
        "blurry, low quality, distorted, watermark, text, logo, SFS, rings, jewelry, extra fingers, deformed anatomy, smooth nails, low resolution"
    ),
    "c7c93112a8dd45edeacfe3496ff8a4c9.jpeg": (
        "a highly detailed, medical realism close up photo of two hands, dark skin tone, psoriatic plaques across knuckles and fingers, inflamed red and scaly lesions, natural fingernails, clear clinical lighting, dermatology photography",
        "blurry, low quality, distorted, watermark, text, logo, SFS, single hand, healthy skin, smooth skin, extra fingers, low resolution"
    ),
}

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)
pipe.safety_checker = None
pipe.enable_model_cpu_offload()

for img_file, (prompt, negative_prompt) in prompts.items():
    img_path = f"/content/images/{img_file}"
    if not os.path.exists(img_path):
        print(f"Skipping {img_file} — not found")
        continue

    # crop bottom 10% to remove SFS watermark, then center crop
    image = load_image(img_path)
    w, h = image.size
    image = image.crop((0, 0, w, int(h * 0.9)))
    w, h = image.size
    crop_size = min(w, h)
    left = (w - crop_size) // 2
    top = (h - crop_size) // 2
    image = image.crop((left, top, left + crop_size, top + crop_size)).resize((512, 512))

    # generate 4 images per seed
    outputs = pipe(
        prompt=prompt,
        image=image,
        negative_prompt=negative_prompt,
        strength=0.5,
        guidance_scale=9.0,
        num_inference_steps=50,
        num_images_per_prompt=4
    ).images

    # save into individual folder and all folder
    img_name = img_file.replace(".jpeg", "")
    img_folder = f"/content/outputs/{version}/{img_name}"
    os.makedirs(img_folder, exist_ok=True)

    for i, output in enumerate(outputs):
        filename = f"{img_name}_gen{i+1}.jpeg"
        output.save(f"{img_folder}/{filename}")
        output.save(f"/content/outputs/{version}/all/{filename}")
        print(f"Saved: {filename}")

    print(f"Done: {img_file}")

print("All images complete!")

In [ ]:
#v6
import torch
import os
from PIL import Image
from diffusers import StableDiffusionImg2ImgPipeline
from diffusers.utils import load_image

version = "v6"
os.makedirs(f"/content/outputs/{version}/all", exist_ok=True)

universal_negative = "jewelry, rings, watch, bracelet, metal, silver, gold, shiny, blurry, low resolution, distorted, unnatural colors, deformed anatomy, extra digits, writing, overlay, text, logo, SFS, watermark"

prompts = {
    "34deac35a907f46e6d43b7bfb39fc00d.jpeg": (
        "A detailed, medical realism close-up photo of the elbow or knee on dark brown skin (Fitzpatrick type V or VI). The skin shows distinct, well-circumscribed, thickened plaques with a striking silvery-white scale. The underlying plaque color is dusky red-violet. Shallow fissures are visible within the scale, and there is clear post-inflammatory hyperpigmentation around the edges of the lesion. Clinical lighting is clear and focused.",
        universal_negative
    ),
    "40c298a68522a302b8a22ebf3395194e.jpeg": (
        "A high-resolution clinical photograph focusing on an arm with medium tan skin (Fitzpatrick type III or IV). It features dense, coalescing plaques of thick, adherent, micaceous (silvery) scale that obscure the underlying red plaque color. Natural body hair is growing through the lesions, adding complexity. The distribution covers a large area of the limb, and there is clear skin adjacent for contrast. The lighting highlights the rough, flaky texture.",
        universal_negative
    ),
    "56783377d95a87b04c0f10ec52e0ffa7.jpeg": (
        "A high-quality medical photograph of a shin with medium tan skin (Fitzpatrick type IV). It displays large, map-like geographical areas where the skin is a deep orange-red. Thick, yellowish-brown hyperkeratotic scales form a thick, protective layer over the reddish plaques, creating a distinct island in a sea pattern. The texture is extremely rough and dry. Clinical, clear lighting.",
        universal_negative
    ),
    "b4b0eeeb1cfc1b7fa8cae06c3bfd3cb8.jpeg": (
        "A macro photo in a clinical setting focusing on the fingertips of three fingers on medium-dark brown skin (Fitzpatrick type IV or V). The natural fingernails are severely affected, showing subungual hyperkeratosis (thickening under the nail), pitting on the nail plate, and clear onycholysis (separation of the nail from the bed). The surrounding paronychial skin is thickened, reddish-violet, and has micaceous scale. The background is a clean white fabric background.",
        universal_negative
    ),
    "c7c93112a8dd45edeacfe3496ff8a4c9.jpeg": (
        "A dorsal view photograph of a pair of hands with dark brown skin (Fitzpatrick type VI) resting symmetrically on a dark surface. The skin on the back of both hands and fingers is covered in multiple reddish-violet papules and small, scaly plaques. The texture is rough and slightly shiny in areas, showing thickening of the skin. The lighting is direct and clear, emphasizing the morphology of the lesions on both hands.",
        universal_negative
    ),
}

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)
pipe.safety_checker = None
pipe.enable_model_cpu_offload()

for img_file, (prompt, negative_prompt) in prompts.items():
    img_path = f"/content/images/{img_file}"
    if not os.path.exists(img_path):
        print(f"Skipping {img_file} — not found")
        continue

    # crop bottom 10% to remove SFS watermark then center crop
    image = load_image(img_path)
    w, h = image.size
    image = image.crop((0, 0, w, int(h * 0.9)))
    w, h = image.size
    crop_size = min(w, h)
    left = (w - crop_size) // 2
    top = (h - crop_size) // 2
    image = image.crop((left, top, left + crop_size, top + crop_size)).resize((512, 512))

    outputs = pipe(
        prompt=prompt,
        image=image,
        negative_prompt=negative_prompt,
        strength=0.6,
        guidance_scale=9.0,
        num_inference_steps=50,
        num_images_per_prompt=4
    ).images

    img_name = img_file.replace(".jpeg", "")
    img_folder = f"/content/outputs/{version}/{img_name}"
    os.makedirs(img_folder, exist_ok=True)

    for i, output in enumerate(outputs):
        filename = f"{img_name}_gen{i+1}.jpeg"
        output.save(f"{img_folder}/{filename}")
        output.save(f"/content/outputs/{version}/all/{filename}")
        print(f"Saved: {filename}")

    print(f"Done: {img_file}")

print("All images complete!")

In [ ]:
#v7
import torch
import os
from PIL import Image
from diffusers import StableDiffusionImg2ImgPipeline
from diffusers.utils import load_image

version = "v7"
os.makedirs(f"/content/outputs/{version}/all", exist_ok=True)

universal_negative = "jewelry, rings, watch, bracelet, metal, silver, gold, shiny, blurry, low resolution, distorted, unnatural colors, deformed anatomy, extra digits, writing, overlay, text, logo, SFS, watermark"

prompts = {
    "34deac35a907f46e6d43b7bfb39fc00d.jpeg": (
        "medical realism close-up of elbow on dark brown skin Fitzpatrick V-VI, multiple well-circumscribed thickened plaques, striking silvery-white micaceous scale, dusky red-violet underlying color, shallow fissures within scale, post-inflammatory hyperpigmentation at lesion edges, clear focused clinical lighting",
        universal_negative
    ),
    "40c298a68522a302b8a22ebf3395194e.jpeg": (
        "clinical photograph of forearm with medium tan skin Fitzpatrick III-IV, dense coalescing plaques, thick adherent micaceous silvery scale obscuring red plaque color, natural body hair growing through lesions, large affected area with clear adjacent skin, rough flaky texture, clinical lighting",
        universal_negative
    ),
    "56783377d95a87b04c0f10ec52e0ffa7.jpeg": (
        "medical photograph of shin with medium tan skin Fitzpatrick IV, large map-like areas of deep orange-red skin, thick yellowish-brown hyperkeratotic scales over reddish plaques, distinct island in a sea pattern, extremely rough dry cracked texture, clear clinical lighting",
        universal_negative
    ),
    "b4b0eeeb1cfc1b7fa8cae06c3bfd3cb8.jpeg": (
        "macro clinical photo of three fingertips on medium-dark brown skin Fitzpatrick IV-V, subungual hyperkeratosis, nail pitting on nail plate, onycholysis visible, thickened reddish-violet paronychial skin with micaceous scale, clean white fabric background, focused clinical lighting",
        universal_negative
    ),
    "c7c93112a8dd45edeacfe3496ff8a4c9.jpeg": (
        "dorsal view of both hands with dark brown skin Fitzpatrick VI resting symmetrically, multiple reddish-violet papules and scaly plaques covering knuckles and fingers, rough thickened skin, slightly shiny in areas, direct clear clinical lighting emphasizing lesion morphology",
        universal_negative
    ),
}

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)
pipe.safety_checker = None
pipe.enable_model_cpu_offload()

for img_file, (prompt, negative_prompt) in prompts.items():
    img_path = f"/content/images/{img_file}"
    if not os.path.exists(img_path):
        print(f"Skipping {img_file} — not found")
        continue

    # crop bottom 10% to remove SFS watermark then center crop
    image = load_image(img_path)
    w, h = image.size
    image = image.crop((0, 0, w, int(h * 0.9)))
    w, h = image.size
    crop_size = min(w, h)
    left = (w - crop_size) // 2
    top = (h - crop_size) // 2
    image = image.crop((left, top, left + crop_size, top + crop_size)).resize((512, 512))

    outputs = pipe(
        prompt=prompt,
        image=image,
        negative_prompt=negative_prompt,
        strength=0.6,
        guidance_scale=9.0,
        num_inference_steps=50,
        num_images_per_prompt=4
    ).images

    img_name = img_file.replace(".jpeg", "")
    img_folder = f"/content/outputs/{version}/{img_name}"
    os.makedirs(img_folder, exist_ok=True)

    for i, output in enumerate(outputs):
        filename = f"{img_name}_gen{i+1}.jpeg"
        output.save(f"{img_folder}/{filename}")
        output.save(f"/content/outputs/{version}/all/{filename}")
        print(f"Saved: {filename}")

    print(f"Done: {img_file}")

print("All images complete!")

In [ ]:
#v8,v9, and v10
import torch
import os
from PIL import Image
from controlnet_aux import HEDdetector
from diffusers import StableDiffusionControlNetImg2ImgPipeline, ControlNetModel, UniPCMultistepScheduler
from diffusers.utils import load_image

os.makedirs("/content/outputs", exist_ok=True)

universal_negative = "jewelry, rings, watch, bracelet, metal, silver, gold, shiny, blurry, low resolution, distorted, unnatural colors, deformed anatomy, extra digits, writing, overlay, text, logo, SFS, watermark"

prompts = {
    "34deac35a907f46e6d43b7bfb39fc00d.jpeg": "medical realism close-up of elbow on dark brown skin Fitzpatrick V-VI, multiple well-circumscribed thickened plaques, striking silvery-white micaceous scale, dusky red-violet underlying color, shallow fissures within scale, post-inflammatory hyperpigmentation at lesion edges",
    "40c298a68522a302b8a22ebf3395194e.jpeg": "clinical photograph of forearm with medium tan skin Fitzpatrick III-IV, dense coalescing plaques, thick adherent micaceous silvery scale obscuring red plaque color, natural body hair growing through lesions, large affected area, rough flaky texture",
    "56783377d95a87b04c0f10ec52e0ffa7.jpeg": "medical photograph of shin with medium tan skin Fitzpatrick IV, large map-like areas of deep orange-red skin, thick yellowish-brown hyperkeratotic scales over reddish plaques, island in a sea pattern, extremely rough dry cracked texture",
    "b4b0eeeb1cfc1b7fa8cae06c3bfd3cb8.jpeg": "macro clinical photo of three fingertips on medium-dark brown skin Fitzpatrick IV-V, subungual hyperkeratosis, nail pitting on nail plate, onycholysis visible, thickened reddish-violet paronychial skin with micaceous scale, white fabric background",
    "c7c93112a8dd45edeacfe3496ff8a4c9.jpeg": "dorsal view of both hands with dark brown skin Fitzpatrick VI resting symmetrically, multiple reddish-violet papules and scaly plaques covering knuckles and fingers, rough thickened skin, direct clear clinical lighting emphasizing lesion morphology",
}

version_configs = [
    {
        "version": "v8",
        "strength": 0.30,
        "guidance_scale": 7.5,
        "num_inference_steps": 50,
        "seed": 42,
        "controlnet_conditioning_scale": 0.8,
        "prompt_suffix": "neutral clinical background, clear lighting"
    },
    {
        "version": "v9",
        "strength": 0.40,
        "guidance_scale": 8.5,
        "num_inference_steps": 50,
        "seed": 123,
        "controlnet_conditioning_scale": 0.7,
        "prompt_suffix": "natural daylight, slight angle variation"
    },
    {
        "version": "v10",
        "strength": 0.50,
        "guidance_scale": 9.0,
        "num_inference_steps": 75,
        "seed": 999,
        "controlnet_conditioning_scale": 0.6,
        "prompt_suffix": "close up macro dermatology photography, high detail"
    },
]

# load models once
processor = HEDdetector.from_pretrained('lllyasviel/Annotators')
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11p_sd15_softedge",
    torch_dtype=torch.float16
)
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.safety_checker = None
pipe.enable_model_cpu_offload()

for config in version_configs:
    version = config["version"]
    strength = config["strength"]
    guidance_scale = config["guidance_scale"]
    num_inference_steps = config["num_inference_steps"]
    seed = config["seed"]
    controlnet_conditioning_scale = config["controlnet_conditioning_scale"]
    prompt_suffix = config["prompt_suffix"]

    os.makedirs(f"/content/outputs/{version}/all", exist_ok=True)
    print(f"\n--- Starting {version} | strength={strength} | guidance={guidance_scale} | controlnet={controlnet_conditioning_scale} | seed={seed} ---\n")

    for img_file, base_prompt in prompts.items():
        img_path = f"/content/images/{img_file}"
        if not os.path.exists(img_path):
            print(f"Skipping {img_file} — not found")
            continue

        # crop bottom 10% to remove SFS watermark then center crop
        image = load_image(img_path)
        w, h = image.size
        image = image.crop((0, 0, w, int(h * 0.9)))
        w, h = image.size
        crop_size = min(w, h)
        left = (w - crop_size) // 2
        top = (h - crop_size) // 2
        image = image.crop((left, top, left + crop_size, top + crop_size)).resize((512, 512))

        # extract softedge map
        softedge = processor(image)

        full_prompt = f"{base_prompt}, {prompt_suffix}"
        generator = torch.Generator().manual_seed(seed)

        outputs = pipe(
            prompt=full_prompt,
            image=image,
            control_image=softedge,
            negative_prompt=universal_negative,
            strength=strength,
            guidance_scale=guidance_scale,
            num_inference_steps=num_inference_steps,
            controlnet_conditioning_scale=controlnet_conditioning_scale,
            num_images_per_prompt=4,
            generator=generator
        ).images

        img_name = img_file.replace(".jpeg", "")
        img_folder = f"/content/outputs/{version}/{img_name}"
        os.makedirs(img_folder, exist_ok=True)

        for i, output in enumerate(outputs):
            filename = f"{img_name}_gen{i+1}.jpeg"
            output.save(f"{img_folder}/{filename}")
            output.save(f"/content/outputs/{version}/all/{filename}")
            print(f"Saved: {filename}")

        print(f"Done: {img_file}")

    print(f"\n--- {version} complete ---\n")

print("All versions complete!")

In [ ]:
#v10,v11
import torch
import os
from PIL import Image
from controlnet_aux import HEDdetector
from diffusers import StableDiffusionControlNetImg2ImgPipeline, ControlNetModel, UniPCMultistepScheduler
from diffusers.utils import load_image

os.makedirs("/content/outputs", exist_ok=True)

universal_negative = "jewelry, rings, watch, bracelet, metal, silver, gold, shiny, blurry, low resolution, distorted, unnatural colors, deformed anatomy, extra digits, writing, overlay, text, logo, SFS, watermark, elbow, knee, forearm, finger, hand, shin"

# ── location-relocated prompts ──────────────────────────────────────────────
# Same disease morphology as originals, new body part, skin tone placeholder {skin}
prompt_templates = {
    "34deac35a907f46e6d43b7bfb39fc00d.jpeg": {
        "location": "lower back",
        "base": "medical realism close-up of lower back on {skin}, multiple well-circumscribed thickened plaques, striking silvery-white micaceous scale, dusky red-violet underlying color, shallow fissures within scale, post-inflammatory hyperpigmentation at lesion edges",
    },
    "40c298a68522a302b8a22ebf3395194e.jpeg": {
        "location": "upper arm and shoulder",
        "base": "clinical photograph of upper arm and shoulder with {skin}, dense coalescing plaques, thick adherent micaceous silvery scale obscuring red plaque color, natural body hair growing through lesions, large affected area, rough flaky texture",
    },
    "56783377d95a87b04c0f10ec52e0ffa7.jpeg": {
        "location": "calf and ankle",
        "base": "medical photograph of calf and ankle with {skin}, large map-like areas of deep orange-red skin, thick yellowish-brown hyperkeratotic scales over reddish plaques, island in a sea pattern, extremely rough dry cracked texture",
    },
    "b4b0eeeb1cfc1b7fa8cae06c3bfd3cb8.jpeg": {
        "location": "toes",
        "base": "macro clinical photo of three toes on {skin}, subungual hyperkeratosis, nail pitting on nail plate, onycholysis visible, thickened reddish-violet paronychial skin with micaceous scale, white fabric background",
    },
    "c7c93112a8dd45edeacfe3496ff8a4c9.jpeg": {
        "location": "back of feet",
        "base": "dorsal view of both feet with {skin} resting symmetrically, multiple reddish-violet papules and scaly plaques covering toe knuckles and dorsum, rough thickened skin, direct clear clinical lighting emphasizing lesion morphology",
    },
}

# ── skin tone definitions ────────────────────────────────────────────────────
# v11: matched to original (same Fitzpatrick as source images)
original_skin = {
    "34deac35a907f46e6d43b7bfb39fc00d.jpeg": "dark brown skin Fitzpatrick V-VI",
    "40c298a68522a302b8a22ebf3395194e.jpeg": "medium tan skin Fitzpatrick III-IV",
    "56783377d95a87b04c0f10ec52e0ffa7.jpeg": "medium tan skin Fitzpatrick IV",
    "b4b0eeeb1cfc1b7fa8cae06c3bfd3cb8.jpeg": "medium-dark brown skin Fitzpatrick IV-V",
    "c7c93112a8dd45edeacfe3496ff8a4c9.jpeg": "dark brown skin Fitzpatrick VI",
}

# v10: shifted darker to add dark skin diversity
shifted_skin = {
    "34deac35a907f46e6d43b7bfb39fc00d.jpeg": "deep black skin Fitzpatrick VI",          # already dark → push deepest
    "40c298a68522a302b8a22ebf3395194e.jpeg": "dark brown skin Fitzpatrick V",             # III-IV → V
    "56783377d95a87b04c0f10ec52e0ffa7.jpeg": "dark brown skin Fitzpatrick V-VI",          # IV → V-VI
    "b4b0eeeb1cfc1b7fa8cae06c3bfd3cb8.jpeg": "dark brown skin Fitzpatrick V-VI",          # IV-V → V-VI
    "c7c93112a8dd45edeacfe3496ff8a4c9.jpeg": "deep black skin Fitzpatrick VI",            # already VI → reinforce
}

# ── version configs (both build on v8 baseline) ──────────────────────────────
version_configs = [
    {
        "version": "v10",
        "skin_map": shifted_skin,
        "strength": 0.55,
        "guidance_scale": 8.5,
        "num_inference_steps": 50,
        "seed": 123,
        "controlnet_conditioning_scale": 0.85,
        "prompt_suffix": "natural daylight, slight angle variation, tight crop showing skin detail",
    },
    {
        "version": "v11",
        "skin_map": original_skin,
        "strength": 0.55,
        "guidance_scale": 8.5,
        "num_inference_steps": 50,
        "seed": 123,
        "controlnet_conditioning_scale": 0.85,
        "prompt_suffix": "natural daylight, slight angle variation, tight crop showing skin detail",
    },
]

# ── load models once ─────────────────────────────────────────────────────────
processor = HEDdetector.from_pretrained('lllyasviel/Annotators')
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11p_sd15_softedge",
    torch_dtype=torch.float16
)
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.safety_checker = None
pipe.enable_model_cpu_offload()

# ── helper: zoom crop to lesion centre ──────────────────────────────────────
def zoom_crop(image, zoom=0.75):
    """
    Centre-crop to `zoom` fraction of the shorter axis so the lesion
    fills the frame, then resize to 512×512.
    A zoom of 0.75 on a 512px source gives a 384px crop → upscaled.
    """
    w, h = image.size
    crop_dim = int(min(w, h) * zoom)
    left  = (w - crop_dim) // 2
    top   = (h - crop_dim) // 2
    return image.crop((left, top, left + crop_dim, top + crop_dim)).resize((512, 512), Image.LANCZOS)

# ── main loop ────────────────────────────────────────────────────────────────
for config in version_configs:
    version   = config["version"]
    skin_map  = config["skin_map"]
    strength  = config["strength"]
    guidance_scale           = config["guidance_scale"]
    num_inference_steps      = config["num_inference_steps"]
    seed                     = config["seed"]
    controlnet_conditioning_scale = config["controlnet_conditioning_scale"]
    prompt_suffix            = config["prompt_suffix"]

    os.makedirs(f"/content/outputs/{version}/all", exist_ok=True)
    print(f"\n{'='*60}")
    print(f"  {version} | strength={strength} | guidance={guidance_scale} | controlnet={controlnet_conditioning_scale} | seed={seed}")
    print(f"{'='*60}\n")

    for img_file, tmpl in prompt_templates.items():
        img_path = f"/content/images/{img_file}"
        if not os.path.exists(img_path):
            print(f"  ⚠ Skipping {img_file} — not found")
            continue

        # load → remove watermark strip → zoom crop centred on lesion
        image = load_image(img_path)
        w, h  = image.size
        image = image.crop((0, 0, w, int(h * 0.9)))   # strip bottom 10% watermark
        image = zoom_crop(image, zoom=0.75)            # tight zoom into lesion

        # build prompt with correct skin tone for this version
        skin        = skin_map[img_file]
        full_prompt = f"{tmpl['base'].format(skin=skin)}, {prompt_suffix}"

        softedge  = processor(image)
        generator = torch.Generator().manual_seed(seed)

        outputs = pipe(
            prompt=full_prompt,
            image=image,
            control_image=softedge,
            negative_prompt=universal_negative,
            strength=strength,
            guidance_scale=guidance_scale,
            num_inference_steps=num_inference_steps,
            controlnet_conditioning_scale=controlnet_conditioning_scale,
            num_images_per_prompt=4,
            generator=generator,
        ).images

        img_name   = img_file.replace(".jpeg", "")
        img_folder = f"/content/outputs/{version}/{img_name}"
        os.makedirs(img_folder, exist_ok=True)

        for i, output in enumerate(outputs):
            filename = f"{img_name}_gen{i+1}.jpeg"
            output.save(f"{img_folder}/{filename}")
            output.save(f"/content/outputs/{version}/all/{filename}")
            print(f"  ✓ Saved: {version}/{img_name}/{filename}")

        print(f"  → Done: {img_file} [{tmpl['location']}]\n")

    print(f"\n{'='*60}")
    print(f"  {version} complete")
    print(f"{'='*60}\n")

print("\n✅ All versions complete!")

In [ ]:
from google.colab import files
import zipfile, os

zip_path = "/content/outputs_v7_v8_v9.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for version in ["v7", "v8", "v9"]:
        for root, dirs, filenames in os.walk(f"/content/outputs/{version}"):
            for filename in filenames:
                filepath = os.path.join(root, filename)
                zipf.write(filepath, os.path.relpath(filepath, "/content/outputs"))

files.download(zip_path)